In [78]:
import tensorflow as tf
from tensorflow import keras
from keras import layers , Sequential

In [79]:
!pip install -q datasets


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [80]:
from datasets import load_dataset

dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-hi"
)

print(dataset)

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 534319
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})


In [81]:
train_data = dataset["train"]

english_sentences = [
    x["translation"]["en"]
    for x in train_data
]

hindi_sentences = [
    x["translation"]["hi"]
    for x in train_data
]

In [82]:
print(english_sentences[:5])
print(hindi_sentences[:5])

['Other, Private Use', '[SCREAMING]', 'Spouse', 'I will never salute you!', 'and the stars and the trees bow themselves;']
['अन्य, निज़ी उपयोग', 'ऊबड़ .', 'जीवनसाथी', '- तुम एक कमांडर कभी नहीं होगा!', 'और तारे और वृक्ष सजदा करते है;']


In [83]:
english_sentences = english_sentences[:50000]
hindi_sentences = hindi_sentences[:50000]

In [84]:
hindi_sentences = [
    "<start> " + sentence + " <end>"
    for sentence in hindi_sentences
]

In [85]:
from sklearn.model_selection import train_test_split

eng_train, eng_test, hin_train, hin_test = train_test_split(
    english_sentences,
    hindi_sentences,
    test_size=0.1,
    random_state=42
)

In [86]:
from tensorflow.keras.preprocessing.text import Tokenizer

eng_tokenizer = Tokenizer(
    num_words = 20000,
    oov_token = "<unk>"
)
eng_tokenizer.fit_on_texts(eng_train)

In [87]:
eng_train_seq = eng_tokenizer.texts_to_sequences(eng_train)
eng_test_seq = eng_tokenizer.texts_to_sequences(eng_test)

In [88]:
print(eng_train[0])
print(eng_train_seq[0])

It's time for my final disappearing act.
[146, 114, 13, 52, 1649, 8801, 813]


In [89]:
hin_tokenizer = Tokenizer(
    num_words = 20000,
    oov_token = "<unk>"
)
hin_tokenizer.fit_on_texts(hin_train)

In [90]:
hin_train_seq = hin_tokenizer.texts_to_sequences(hin_train)
hin_test_seq = hin_tokenizer.texts_to_sequences(hin_test)

In [91]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

hin_train_seq = pad_sequences(
    hin_train_seq,
    padding = "post"
)

hin_test_seq = pad_sequences(
    hin_test_seq,
    padding = "post",
    maxlen = hin_train_seq.shape[1]
)


eng_train_seq = pad_sequences(
    eng_train_seq,
    padding = "post"
)

eng_test_seq = pad_sequences(
    eng_test_seq,
    padding = "post",
    maxlen = eng_train_seq.shape[1]
)

In [92]:
decoder_input = hin_train_seq[ : , : -1]
decoder_target = hin_train_seq[ : , 1 : ]

decoder_test_input = hin_test_seq[:, :-1]
decoder_test_target = hin_test_seq[:, 1:]

# Encoder

In [93]:
eng_vocab_size = len(eng_tokenizer.word_index) + 1
hin_vocab_size = len(hin_tokenizer.word_index) + 1

In [94]:
embedding_dim = 256
latent_dim  = 256

In [95]:
encoder_inputs = layers.Input(
    shape = (None , )
)

encoder_embedding = layers.Embedding(
    eng_vocab_size,
    embedding_dim
)(encoder_inputs)

encoder_lstm = layers.LSTM(
    latent_dim,
    return_state = True
)

encoder_outputs , state_h , state_c = encoder_lstm(
    encoder_embedding
)

# Decoder

In [96]:
decoder_inputs = layers.Input(
    shape = (None , )
)

decoder_embedding = layers.Embedding(
    hin_vocab_size , 
    embedding_dim
)(decoder_inputs)

decoder_lstm = layers.LSTM(
    latent_dim,
    return_sequences = True,
    return_state = True
)

decoder_outputs , _ , _ = decoder_lstm(
    decoder_embedding,
    initial_state = [state_h , state_c]
)

# Dense Layers

In [97]:
decoder_dense = layers.Dense(
    hin_vocab_size,
    activation = "softmax"
)

decoder_output = decoder_dense(
    decoder_outputs
)

## Encoder-Encoder Output

In [98]:
model = tf.keras.Model(
    [encoder_inputs , decoder_inputs],
    decoder_output
)

model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, None, 256) │  5,478,912 │ input_layer_6[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, None, 256) │  7,074,816 │ input_layer_7[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, 256),     │    525,312 │ embedding_6[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_6 (LSTM)       │ [(None, None,     │    525,312 │ embedding_7[0][0… │
│                     │ 256), (None,      │            │ lstm_5[0][1],     │
│                     │ 256), (None,      │            │ lstm_5[0][2]      │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, None,      │  7,102,452 │ lstm_6[0][0]      │
│                     │ 27636)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 20,706,804 (78.99 MB)

 Trainable params: 20,706,804 (78.99 MB)

 Non-trainable params: 0 (0.00 B)

In [99]:
model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

In [ ]:
history = model.fit(
    [eng_train_seq, decoder_input],
    decoder_target,
    validation_data=(
        [eng_test_seq, decoder_test_input],
        decoder_test_target
    ),
    batch_size=64,
    epochs=20
)